# Late Delivery Prediction: Data and Models

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from scipy.stats import chi2_contingency, mannwhitneyu

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / '.env').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / '.env').is_file():
    raise FileNotFoundError('Could not locate .env. Start Jupyter from the project root or notebooks/.')

SOURCE_DIR = PROJECT_ROOT / 'src'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from ml.train_model import ALERT_FRACTION, evaluate_probabilities, build_pipeline, split_features

load_dotenv(PROJECT_ROOT / '.env')
required_settings = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError('Missing database settings: ' + ', '.join(missing_settings))

NUMERIC_FEATURES = ['promised_lead_time_days', 'item_count', 'total_order_value']
STATISTICAL_CATEGORICAL_FEATURES = ['purchase_month', 'purchase_weekday', 'customer_state', 'has_out_of_state_seller']
MODEL_CATEGORICAL_FEATURES = ['purchase_month', 'customer_state', 'has_out_of_state_seller']
STATISTICAL_FEATURES = NUMERIC_FEATURES + STATISTICAL_CATEGORICAL_FEATURES
MODEL_FEATURES = NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES


## Data and Splits

In [2]:
FEATURE_QUERY = '''
SELECT
    order_id,
    purchase_date_key,
    purchase_month,
    purchase_weekday,
    promised_lead_time_days,
    customer_state,
    has_out_of_state_seller,
    item_count,
    total_order_value,
    is_ml_eligible,
    is_timeliness_eligible,
    is_late
FROM mart.order_delivery_ml_features
ORDER BY purchase_date_key, order_id
'''

with psycopg.connect(
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(FEATURE_QUERY)
        columns = [column.name for column in cursor.description]
        features = pd.DataFrame(cursor.fetchall(), columns=columns)

features['purchase_date'] = pd.to_datetime(
    features['purchase_date_key'].astype(str), format='%Y%m%d', errors='raise'
)
for column in NUMERIC_FEATURES:
    features[column] = pd.to_numeric(features[column], errors='coerce')
for column in ('is_ml_eligible', 'is_timeliness_eligible', 'is_late'):
    features[column] = features[column].astype('boolean')
features['has_out_of_state_seller'] = features['has_out_of_state_seller'].map(
    {True: 'yes', False: 'no'}
).astype(object)

labelled = features.loc[
    features['is_ml_eligible']
    & features['is_timeliness_eligible']
    & features['is_late'].notna()
].copy()
if labelled.empty:
    raise ValueError('No eligible labelled rows. Check the ML feature mart pipeline.')

ordered = labelled.sort_values(['purchase_date', 'order_id']).reset_index(drop=True)
train, validation, test = split_features(ordered)

def split_row(name, frame):
    return {
        'split': name,
        'orders': len(frame),
        'first_purchase': frame['purchase_date'].min(),
        'last_purchase': frame['purchase_date'].max(),
        'late_rate': frame['is_late'].mean(),
    }

population_summary = pd.DataFrame({
    'orders_in_mart': [len(features)],
    'eligible_labelled_orders': [len(labelled)],
    'late_rate': [labelled['is_late'].mean()],
})
split_summary = pd.DataFrame([
    split_row('train', train),
    split_row('validation', validation),
    split_row('test', test),
])
display(population_summary, split_summary)


   orders_in_mart  eligible_labelled_orders  late_rate
0           99441                     96450      0.068         split  orders first_purchase last_purchase  late_rate
0       train   67515     2016-09-15    2018-04-15      0.078
1  validation   14467     2018-04-15    2018-06-21      0.043
2        test   14468     2018-06-21    2018-08-29      0.043


## Training-Set Checks

In [3]:
def outlier_summary(frame, columns):
    rows = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors='coerce').dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        rows.append({
            'feature': column,
            'min': values.min(),
            'median': values.median(),
            'max': values.max(),
            'outside_iqr': int(((values < lower) | (values > upper)).sum()),
        })
    return pd.DataFrame(rows).set_index('feature')

def calculate_vif(frame, columns):
    numeric = frame[columns].apply(pd.to_numeric, errors='coerce')
    numeric = numeric.fillna(numeric.median())
    rows = []
    for column in columns:
        target = numeric[column].to_numpy()
        others = numeric.drop(columns=column).to_numpy()
        total_sum_squares = np.square(target - target.mean()).sum()
        if np.isclose(total_sum_squares, 0):
            vif = np.inf
        else:
            design = np.column_stack([np.ones(len(target)), others])
            coefficients, *_ = np.linalg.lstsq(design, target, rcond=None)
            residual_sum_squares = np.square(target - design @ coefficients).sum()
            r_squared = 1 - residual_sum_squares / total_sum_squares
            vif = np.inf if 1 - r_squared < 1e-10 else 1 / (1 - r_squared)
        rows.append({'feature': column, 'vif': vif})
    return pd.DataFrame(rows).sort_values('vif', ascending=False)

def grouped_categories(series, minimum_count=30):
    values = series.astype('string').fillna('<missing>')
    rare = values.value_counts()[lambda counts: counts < minimum_count].index
    return values.where(~values.isin(rare), '<other>')

missing_profile = train[STATISTICAL_FEATURES].isna().agg(['sum', 'mean']).T.rename(
    columns={'sum': 'missing_rows', 'mean': 'missing_rate'}
)
numeric_by_label = train.groupby('is_late', observed=True)[NUMERIC_FEATURES].agg(['count', 'median', 'mean']).T

mann_whitney_rows = []
for column in NUMERIC_FEATURES:
    late_values = pd.to_numeric(train.loc[train['is_late'].eq(True), column], errors='coerce').dropna()
    non_late_values = pd.to_numeric(train.loc[train['is_late'].eq(False), column], errors='coerce').dropna()
    statistic, p_value = mannwhitneyu(late_values, non_late_values, alternative='two-sided')
    mann_whitney_rows.append({
        'feature': column,
        'late_median': late_values.median(),
        'non_late_median': non_late_values.median(),
        'p_value': p_value,
    })
mann_whitney_results = pd.DataFrame(mann_whitney_rows).set_index('feature').sort_values('p_value')

chi_square_rows = []
for column in STATISTICAL_CATEGORICAL_FEATURES:
    contingency = pd.crosstab(grouped_categories(train[column]), train['is_late'])
    chi_square, p_value, degrees_of_freedom, expected = chi2_contingency(contingency)
    phi_squared = chi_square / expected.sum()
    denominator = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    chi_square_rows.append({
        'feature': column,
        'p_value': p_value,
        'cramers_v': np.sqrt(phi_squared / denominator) if denominator else np.nan,
        'minimum_expected_count': expected.min(),
    })
chi_square_results = pd.DataFrame(chi_square_rows).set_index('feature').sort_values('p_value')

display(missing_profile, numeric_by_label, outlier_summary(train, NUMERIC_FEATURES))
display(calculate_vif(train, NUMERIC_FEATURES), mann_whitney_results, chi_square_results)


                         missing_rows  missing_rate
promised_lead_time_days         0.000         0.000
item_count                      0.000         0.000
total_order_value               0.000         0.000
purchase_month                  0.000         0.000
purchase_weekday                0.000         0.000
customer_state                  0.000         0.000
has_out_of_state_seller         0.000         0.000 is_late                             False     True 
promised_lead_time_days count  62,228.000 5,287.000
                        median     23.380    23.020
                        mean       24.230    23.569
item_count              count  62,228.000 5,287.000
                        median      1.000     1.000
                        mean        1.144     1.114
total_order_value       count  62,228.000 5,287.000
                        median    103.065   115.880
                        mean      156.497   176.931                            min  median        max  outside_iqr
f

## Training and Evaluation

Logistic Regression fits the training period only. The validation period sets the alert threshold, and the test period evaluates the fitted model at that fixed threshold.


In [4]:
pipeline = build_pipeline()
pipeline.fit(train[MODEL_FEATURES], train['is_late'].astype(int))
validation_probabilities = pipeline.predict_proba(validation[MODEL_FEATURES])[:, 1]
y_validation = validation['is_late'].astype(int)
validation_results = pd.DataFrame([
    {'model': 'Logistic Regression', **evaluate_probabilities(y_validation, validation_probabilities, order_ids=validation['order_id'].to_numpy())},
]).set_index('model')

display(validation_results)


                     pr_auc  roc_auc  brier_score  recall_at_top_10_pct
model                                                                  
Logistic Regression   0.109    0.720        0.040                 0.319


In [5]:
alert_threshold = float(np.quantile(validation_probabilities, 1 - ALERT_FRACTION))
X_test, y_test = test[MODEL_FEATURES], test['is_late'].astype(int)
test_probabilities = pipeline.predict_proba(X_test)[:, 1]

final_results = pd.DataFrame([
    {'split': 'validation', **evaluate_probabilities(y_validation, validation_probabilities, alert_threshold, validation['order_id'].to_numpy())},
    {'split': 'test', **evaluate_probabilities(y_test, test_probabilities, alert_threshold, test['order_id'].to_numpy())},
]).set_index('split')

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
model = pipeline.named_steps['model']
feature_effects = pd.DataFrame({'feature': feature_names, 'coefficient': model.coef_[0]})
feature_effects['absolute_effect'] = feature_effects['coefficient'].abs()
feature_effects = feature_effects.sort_values('absolute_effect', ascending=False).drop(columns='absolute_effect')

print('Alert threshold:', alert_threshold)
display(final_results, feature_effects)


Alert threshold: 0.07142805877590301
            pr_auc  roc_auc  brier_score  recall_at_top_10_pct  \
split                                                            
validation   0.109    0.720        0.040                 0.319   
test         0.075    0.643        0.041                 0.180   

            flagged_orders  precision_at_threshold  recall_at_threshold  
split                                                                    
validation            1447                   0.138                0.319  
test                  1848                   0.072                0.215                                        feature  coefficient
16            categorical__customer_state_AL        1.287
32            categorical__customer_state_PR       -1.239
42   categorical__has_out_of_state_seller_no       -1.217
25            categorical__customer_state_MG       -1.151
40            categorical__customer_state_SP       -1.120
21            categorical__customer_state_DF       -1.